## Training on pysprck library with json files 

In [1]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as func

In [2]:
sc = SparkContext('local') # work my pc as distrbuted system
spark = SparkSession(sc) # create session ..

In [51]:
main_df = spark.read.format('json').option('inferSchema' , True)\
    .option('multiLine',True).load([f'day{i}.json' for i in range(1,4)])  # read files on dataFrame ..

In [52]:
main_df.show()

+--------------------+--------------------+--------------------+--------+--------------------+--------------------+
|            customer|               items|            metadata|order_id|             payment|           timestamp|
+--------------------+--------------------+--------------------+--------+--------------------+--------------------+
|{{Calgary, Canada...|[{I103, 199.99, N...|[{referrer, insta...| ORD1003|{Debit Card, TXN7...|2025-06-01T11:00:00Z|
|{{Toronto, Canada...|[{I100, 25.99, Wi...|[{campaign, back_...| ORD1001|{Credit Card, TXN...|2025-06-01T10:15:00Z|
|{{Vancouver, Cana...|[{I102, 45.0, Blu...|[{campaign, cyber...| ORD1002|   {PayPal, TXN7891}|2025-06-01T10:30:00Z|
+--------------------+--------------------+--------------------+--------+--------------------+--------------------+



In [53]:
main_df.collect()
main_df.printSchema()

root
 |-- customer: struct (nullable = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |    |-- postal_code: string (nullable = true)
 |    |-- customer_id: long (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- product_name: string (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |-- metadata: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- key: string (nullable = true)
 |    |    |-- value: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- method: string (nullable = true)
 |    |-- transaction_id: string (nullable = true)


In [ ]:
main_df = main_df.withColumn('items',func.explode_outer('items'))\
    .withColumn('metadata',func.explode_outer('metadata'))
df = main_df.select('customer.name','order_id','customer.address.country','customer.address.city','customer.address.postal_code'
                    ,'customer.customer_id','customer.email','items.item_id','items.product_name',
                    'items.price','items.quantity','payment.method','payment.transaction_id','*')\
                        .drop('customer','metadata','items','payment')

df.show()

+-----------+--------+-------+---------+-----------+-----------+-----------------+-------+--------------------+------+--------+-----------+--------------+--------+--------------------+
|       name|order_id|country|     city|postal_code|customer_id|            email|item_id|        product_name| price|quantity|     method|transaction_id|order_id|           timestamp|
+-----------+--------+-------+---------+-----------+-----------+-----------------+-------+--------------------+------+--------+-----------+--------------+--------+--------------------+
|  David Lee| ORD1003| Canada|  Calgary|    T2P 1G1|        503|david@example.com|   I103|Noise Cancelling ...|199.99|       1| Debit Card|       TXN7892| ORD1003|2025-06-01T11:00:00Z|
|  David Lee| ORD1003| Canada|  Calgary|    T2P 1G1|        503|david@example.com|   I103|Noise Cancelling ...|199.99|       1| Debit Card|       TXN7892| ORD1003|2025-06-01T11:00:00Z|
|  David Lee| ORD1003| Canada|  Calgary|    T2P 1G1|        503|david@examp

In [63]:
df = df.drop_duplicates()
df.show()

+-----------+--------+-------+---------+-----------+-----------+-----------------+-------+--------------------+------+--------+-----------+--------------+--------+--------------------+
|       name|order_id|country|     city|postal_code|customer_id|            email|item_id|        product_name| price|quantity|     method|transaction_id|order_id|           timestamp|
+-----------+--------+-------+---------+-----------+-----------+-----------------+-------+--------------------+------+--------+-----------+--------------+--------+--------------------+
|  David Lee| ORD1003| Canada|  Calgary|    T2P 1G1|        503|david@example.com|   I103|Noise Cancelling ...|199.99|       1| Debit Card|       TXN7892| ORD1003|2025-06-01T11:00:00Z|
|   John Doe| ORD1001| Canada|  Toronto|    M5H 2N2|        501| john@example.com|   I101|       USB-C Adapter| 15.49|       1|Credit Card|       TXN7890| ORD1001|2025-06-01T10:15:00Z|
|  David Lee| ORD1003| Canada|  Calgary|    T2P 1G1|        503|david@examp